<a href="https://colab.research.google.com/github/tartiwiaulia/pcos-detection/blob/dev/pcos_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
# ============================================================
# CELL 1
# Tujuan: load dataset PCOS dari Google Drive ke Colab
# ============================================================
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

df = pd.read_excel(
    '/content/drive/MyDrive/pcos-project/PCOS_data_without_infertility.xlsx',
    sheet_name=1
)

print("Jumlah data:", df.shape)
print("\nKolom yang tersedia:")
print(df.columns.tolist())
print("\nLima data pertama:")
print(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Jumlah data: (541, 45)

Kolom yang tersedia:
['Sl. No', 'Patient File No.', 'PCOS (Y/N)', ' Age (yrs)', 'Weight (Kg)', 'Height(Cm) ', 'BMI', 'Blood Group', 'Pulse rate(bpm) ', 'RR (breaths/min)', 'Hb(g/dl)', 'Cycle(R/I)', 'Cycle length(days)', 'Marraige Status (Yrs)', 'Pregnant(Y/N)', 'No. of aborptions', '  I   beta-HCG(mIU/mL)', 'II    beta-HCG(mIU/mL)', 'FSH(mIU/mL)', 'LH(mIU/mL)', 'FSH/LH', 'Hip(inch)', 'Waist(inch)', 'Waist:Hip Ratio', 'TSH (mIU/L)', 'AMH(ng/mL)', 'PRL(ng/mL)', 'Vit D3 (ng/mL)', 'PRG(ng/mL)', 'RBS(mg/dl)', 'Weight gain(Y/N)', 'hair growth(Y/N)', 'Skin darkening (Y/N)', 'Hair loss(Y/N)', 'Pimples(Y/N)', 'Fast food (Y/N)', 'Reg.Exercise(Y/N)', 'BP _Systolic (mmHg)', 'BP _Diastolic (mmHg)', 'Follicle No. (L)', 'Follicle No. (R)', 'Avg. F size (L) (mm)', 'Avg. F size (R) (mm)', 'Endometrium (mm)', 'Unnamed: 44']

Lima data pertama:
   Sl. No

In [44]:
# ============================================================
# CELL 2
# Tujuan: bersihkan nama kolom dari spasi aneh di awal/akhir
# ============================================================
df.columns = df.columns.str.strip()

print("Kolom setelah dibersihkan:")
print(df.columns.tolist())

Kolom setelah dibersihkan:
['Sl. No', 'Patient File No.', 'PCOS (Y/N)', 'Age (yrs)', 'Weight (Kg)', 'Height(Cm)', 'BMI', 'Blood Group', 'Pulse rate(bpm)', 'RR (breaths/min)', 'Hb(g/dl)', 'Cycle(R/I)', 'Cycle length(days)', 'Marraige Status (Yrs)', 'Pregnant(Y/N)', 'No. of aborptions', 'I   beta-HCG(mIU/mL)', 'II    beta-HCG(mIU/mL)', 'FSH(mIU/mL)', 'LH(mIU/mL)', 'FSH/LH', 'Hip(inch)', 'Waist(inch)', 'Waist:Hip Ratio', 'TSH (mIU/L)', 'AMH(ng/mL)', 'PRL(ng/mL)', 'Vit D3 (ng/mL)', 'PRG(ng/mL)', 'RBS(mg/dl)', 'Weight gain(Y/N)', 'hair growth(Y/N)', 'Skin darkening (Y/N)', 'Hair loss(Y/N)', 'Pimples(Y/N)', 'Fast food (Y/N)', 'Reg.Exercise(Y/N)', 'BP _Systolic (mmHg)', 'BP _Diastolic (mmHg)', 'Follicle No. (L)', 'Follicle No. (R)', 'Avg. F size (L) (mm)', 'Avg. F size (R) (mm)', 'Endometrium (mm)', 'Unnamed: 44']


In [45]:
# ============================================================
# CELL 3 BARU
# Tujuan: tambah fitur dari dataset yang relevan secara klinis
# tetap pilih yang bisa dijawab user tanpa tes lab
# ============================================================

kolom_pakai = [
    'BMI',                    # indeks massa tubuh
    'Age (yrs)',              # usia pasien
    'Cycle(R/I)',             # siklus teratur/tidak
    'Weight gain(Y/N)',       # kenaikan berat badan
    'hair growth(Y/N)',       # pertumbuhan rambut berlebih
    'Pimples(Y/N)',           # jerawat berlebihan
    'Hair loss(Y/N)',         # rambut rontok
    'Skin darkening (Y/N)',   # kulit menghitam
    'Fast food (Y/N)',        # sering makan fast food
    'Reg.Exercise(Y/N)',      # olahraga rutin
    'PCOS (Y/N)'              # label target
]

df_clean = df[kolom_pakai].copy()
df_clean = df_clean.fillna(df_clean.mean(numeric_only=True))

print("Jumlah data:", df_clean.shape)
print("\nNilai kosong per kolom:")
print(df_clean.isnull().sum())
print("\nContoh data:")
print(df_clean.head())

Jumlah data: (541, 11)

Nilai kosong per kolom:
BMI                     0
Age (yrs)               0
Cycle(R/I)              0
Weight gain(Y/N)        0
hair growth(Y/N)        0
Pimples(Y/N)            0
Hair loss(Y/N)          0
Skin darkening (Y/N)    0
Fast food (Y/N)         0
Reg.Exercise(Y/N)       0
PCOS (Y/N)              0
dtype: int64

Contoh data:
         BMI  Age (yrs)  Cycle(R/I)  Weight gain(Y/N)  hair growth(Y/N)  \
0  19.300000         28           2                 0                 0   
1  24.921163         36           2                 0                 0   
2  25.270891         33           2                 0                 0   
3  29.674945         37           2                 0                 0   
4  20.060954         25           2                 0                 0   

   Pimples(Y/N)  Hair loss(Y/N)  Skin darkening (Y/N)  Fast food (Y/N)  \
0             0               0                     0              1.0   
1             0               0         

In [46]:
# ============================================================
# CELL 4
# Tujuan: cek statistik dan distribusi data
# pastikan nilai masuk akal sebelum lanjut training
# ============================================================
print(df_clean.describe())

print("\nDistribusi PCOS (Y/N):")
print(df_clean['PCOS (Y/N)'].value_counts())

print("\nNilai unik Cycle(R/I):")
print(sorted(df_clean['Cycle(R/I)'].unique()))

              BMI   Age (yrs)  Cycle(R/I)  Weight gain(Y/N)  hair growth(Y/N)  \
count  541.000000  541.000000  541.000000        541.000000        541.000000   
mean    24.311285   31.430684    2.560074          0.377079          0.273567   
std      4.056399    5.411006    0.901950          0.485104          0.446202   
min     12.417882   20.000000    2.000000          0.000000          0.000000   
25%     21.641274   28.000000    2.000000          0.000000          0.000000   
50%     24.238227   31.000000    2.000000          0.000000          0.000000   
75%     26.634958   35.000000    4.000000          1.000000          1.000000   
max     38.900000   48.000000    5.000000          1.000000          1.000000   

       Pimples(Y/N)  Hair loss(Y/N)  Skin darkening (Y/N)  Fast food (Y/N)  \
count    541.000000      541.000000            541.000000       541.000000   
mean       0.489834        0.452865              0.306839         0.514815   
std        0.500359        0.498234 

In [47]:
# ============================================================
# CELL 5 BARU
# Tujuan: perbaiki Cycle(R/I) jadi binary 0 dan 1
# ============================================================

kolom_final = [
    'BMI',
    'Age (yrs)',
    'Cycle(R/I)',
    'Weight gain(Y/N)',
    'hair growth(Y/N)',
    'Pimples(Y/N)',
    'Hair loss(Y/N)',
    'Skin darkening (Y/N)',
    'Fast food (Y/N)',
    'Reg.Exercise(Y/N)',
    'PCOS (Y/N)'
]

df_final = df[kolom_final].copy()

df_final['Cycle(R/I)'] = df_final['Cycle(R/I)'].apply(
    lambda x: 1 if x == 4 else 0
)

df_final = df_final.fillna(df_final.mean(numeric_only=True))

print("Jumlah data:", df_final.shape)
print("\nNilai kosong (harus semua 0):")
print(df_final.isnull().sum())
print("\nContoh data:")
print(df_final.head())

Jumlah data: (541, 11)

Nilai kosong (harus semua 0):
BMI                     0
Age (yrs)               0
Cycle(R/I)              0
Weight gain(Y/N)        0
hair growth(Y/N)        0
Pimples(Y/N)            0
Hair loss(Y/N)          0
Skin darkening (Y/N)    0
Fast food (Y/N)         0
Reg.Exercise(Y/N)       0
PCOS (Y/N)              0
dtype: int64

Contoh data:
         BMI  Age (yrs)  Cycle(R/I)  Weight gain(Y/N)  hair growth(Y/N)  \
0  19.300000         28           0                 0                 0   
1  24.921163         36           0                 0                 0   
2  25.270891         33           0                 0                 0   
3  29.674945         37           0                 0                 0   
4  20.060954         25           0                 0                 0   

   Pimples(Y/N)  Hair loss(Y/N)  Skin darkening (Y/N)  Fast food (Y/N)  \
0             0               0                     0              1.0   
1             0               0   

In [48]:
# ============================================================
# CELL 6
# Tujuan: cek berapa data dengan nilai Cycle = 5
# hasilnya cuma 1 data, diabaikan, lanjut training
# ============================================================
print("Jumlah data dengan Cycle(R/I) asli = 5:")
print(len(df[df['Cycle(R/I)'] == 5]))

print("\nDistribusi PCOS untuk Cycle=5:")
print(df[df['Cycle(R/I)'] == 5]['PCOS (Y/N)'].value_counts())

Jumlah data dengan Cycle(R/I) asli = 5:
1

Distribusi PCOS untuk Cycle=5:
PCOS (Y/N)
1    1
Name: count, dtype: int64


In [49]:
# ============================================================
# CELL 7 BARU
# Tujuan: install SMOTE untuk tangani data tidak seimbang
# ============================================================
!pip install imbalanced-learn --quiet

print("Library berhasil diinstall!")

Library berhasil diinstall!


In [50]:
# ============================================================
# CELL 8 BARU
# Tujuan: split data dulu SEBELUM SMOTE
# SMOTE hanya boleh diterapkan ke data training, bukan test
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df_final.drop('PCOS (Y/N)', axis=1)
y = df_final['PCOS (Y/N)']

# split 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# normalisasi data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Distribusi sebelum SMOTE:")
import pandas as pd
print(pd.Series(y_train).value_counts())

Distribusi sebelum SMOTE:
PCOS (Y/N)
0    287
1    145
Name: count, dtype: int64


In [51]:
# ============================================================
# CELL 9 BARU
# Tujuan: terapkan SMOTE ke data training
# membuat data sintetis PCOS positif supaya seimbang
# ============================================================
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Distribusi setelah SMOTE:")
print(pd.Series(y_train_sm).value_counts())

Distribusi setelah SMOTE:
PCOS (Y/N)
1    287
0    287
Name: count, dtype: int64


In [52]:
# ============================================================
# CELL 10 BARU
# Tujuan: cari kombinasi hyperparameter JST terbaik
# pakai GridSearchCV untuk test banyak kombinasi sekaligus
# ============================================================
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

# daftar kombinasi yang akan dicoba otomatis
param_grid = {
    'hidden_layer_sizes': [
        (128, 64, 32),
        (256, 128, 64),
        (128, 64),
        (256, 128)
    ],
    'activation': ['relu', 'tanh'],
    'solver': ['adam'],
    'max_iter': [2000],
    'learning_rate': ['adaptive'],
    'alpha': [0.0001, 0.001, 0.01]
}

# GridSearch test semua kombinasi pakai cross validation 5 fold
grid_search = GridSearchCV(
    MLPClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_sm, y_train_sm)

print("Kombinasi terbaik:")
print(grid_search.best_params_)

# evaluasi model terbaik di data test asli
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\nAkurasi model JST:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print("\nLaporan lengkap:")
print(classification_report(y_test, y_pred,
      target_names=['PCOS Negatif', 'PCOS Positif']))

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Kombinasi terbaik:
{'activation': 'relu', 'alpha': 0.01, 'hidden_layer_sizes': (128, 64, 32), 'learning_rate': 'adaptive', 'max_iter': 2000, 'solver': 'adam'}

Akurasi model JST: 79.82 %

Laporan lengkap:
              precision    recall  f1-score   support

PCOS Negatif       0.84      0.88      0.86        77
PCOS Positif       0.68      0.59      0.63        32

    accuracy                           0.80       109
   macro avg       0.76      0.74      0.75       109
weighted avg       0.79      0.80      0.79       109



In [54]:
# ============================================================
# CELL 10B — satu kombinasi terakhir
# berdasarkan hasil GridSearch, coba alpha lebih kecil
# dengan layer yang sama (128, 64, 32)
# ============================================================
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

model_final = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    alpha=0.0001,          # lebih kecil dari 0.01
    max_iter=3000,         # lebih banyak iterasi
    learning_rate='adaptive',
    early_stopping=True,   # stop kalau tidak ada progress
    validation_fraction=0.1,
    random_state=42
)

model_final.fit(X_train_sm, y_train_sm)
y_pred = model_final.predict(X_test)

print("Akurasi model JST final:",
      round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print("\nLaporan lengkap:")
print(classification_report(y_test, y_pred,
      target_names=['PCOS Negatif', 'PCOS Positif']))

Akurasi model JST final: 86.24 %

Laporan lengkap:
              precision    recall  f1-score   support

PCOS Negatif       0.89      0.92      0.90        77
PCOS Positif       0.79      0.72      0.75        32

    accuracy                           0.86       109
   macro avg       0.84      0.82      0.83       109
weighted avg       0.86      0.86      0.86       109



In [56]:
# ============================================================
# CELL 11
# Tujuan: simpan model_final yang akurasi 86.24% ke Drive
# ============================================================
import pickle
import shutil

# simpan model_final bukan model lama
with open('pcos_model.pkl', 'wb') as f:
    pickle.dump(model_final, f)

with open('pcos_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

shutil.copy('pcos_model.pkl',
    '/content/drive/MyDrive/pcos-project/pcos_model.pkl')

shutil.copy('pcos_scaler.pkl',
    '/content/drive/MyDrive/pcos-project/pcos_scaler.pkl')

print("Model final 86.24% tersimpan di Google Drive!")

Model final 86.24% tersimpan di Google Drive!
